## IMPORT LIBRARIES, SET CONFIGURATIONS AND LOAD DATA FROM FILES INTO DATAFRAMES

### IMPORT LIBRARIES

In [ ]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

### SET CONFIGURATION

In [ ]:
pd.set_option("display.max_columns", None)

### GET USER INPUT AND TIME SERIES DATA

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputDataRaw = loadDataFromFile("User:Previous experiments")
timeSeriesData = loadDataFromFile("Data:Previous experiments")

In [ ]:
userInputDataRaw

In [ ]:
userInputDataRaw["room"].unique()

In [ ]:
timeSeriesData

In [ ]:
userInputData = userInputDataRaw.drop(columns=["_id","timestamp","epoch_ms","userInputCategory","pollutant-type","quantity-used"])

## PROCCESS THE TIME COLUMNS AND VOC COLUMNS

### GET THE TIMEDATA FOR EACH EXPERIMENT,COMPARE STARTING EXPERIMENT STATE ROWS AND REMOVING SOURCE POLLUTANT STATE ROWS INTO THE IN BETWEEN INSERTING SOURCE POLLUTANT ROWS 

In [ ]:
userInputData["timestamp_local"] = pd.to_datetime(userInputData["timestamp_local"])  # ensure datetime
userInputData["timestamp_local"] = userInputData["timestamp_local"].dt.floor("s")

In [ ]:
userInputData

In [ ]:
userInputDataTemp = userInputData[userInputData["experimentState"].isin(["NoSourcePollutantInserted","InsertingSourcePollutant"])].copy()

userInputDataTemp["timestamp StartingExperiment"] = userInputData["timestamp_local"].shift(1).loc[userInputData.index]
userInputDataTemp["timestamp EndingExperiment"]= userInputData["timestamp_local"].shift(-1).loc[userInputData.index]
userInputData =userInputDataTemp
userInputData = userInputData.reset_index(drop = True)
userInputData = userInputData.rename(columns={"timestamp_local":"timestamp InsertingSource"})



In [ ]:
userInputData.head(20)

In [ ]:
dict_of_timeseries = {}
for index, row in userInputData.iterrows():
    #get datetime
    start = row["timestamp StartingExperiment"]
    turning_point = row["timestamp InsertingSource"]
    end = row["timestamp EndingExperiment"]
    #Grab data
    particular_experiment = timeSeriesData[(timeSeriesData["timestamp"] >= start) & (timeSeriesData["timestamp"] <= end)]
    particular_experiment = particular_experiment.sort_values(by = ["timestamp"],ignore_index = True)
    dict_of_timeseries[index] = particular_experiment



In [ ]:
userInputData["date of experiment"] = userInputData["timestamp EndingExperiment"].apply(lambda x:x.date())

### PROCESS THE EXPERIMENTS OF EACH ELEMENT OF DICTIONARY,IN ORDER TO INTERPOLATE THE DATA INTO A PER SECOND FREQUENCY

In [ ]:
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index]

    #make the dataframe from multiple columns to have the data for each sensor to just one column
    particular_experiment = particular_experiment.melt(
        id_vars=["timestamp"], 
        value_vars=[
            "Id=0:BME680:breathVocEquivalent",
            "Id=1:BME680:breathVocEquivalent",
            "Id=2:BME680:breathVocEquivalent"
        ],
        var_name="sensors", 
        value_name="VOC"
    )
    particular_experiment = particular_experiment.dropna(ignore_index = True)

    particular_experiment = particular_experiment.sort_values(by = ["timestamp"],ignore_index = True)
    dict_of_timeseries[index] = particular_experiment

In [ ]:
example_df = dict_of_timeseries[10]
example_df.head(20)

In [ ]:
userInputData

In [ ]:
for index, row in userInputData.iterrows():
#Now, we only have timestamps based of whenever the sensor took the value, but we want to have times 
    # Get full range of seconds from start to end timestamp
    particular_experiment =  dict_of_timeseries[index]
    start = row["timestamp StartingExperiment"]
    turning_point = row["timestamp InsertingSource"]

    end = row["timestamp EndingExperiment"]
    full_time_range = pd.date_range(start, end, freq="s")
    
    all_sensors = particular_experiment["sensors"].unique()
    particular_experiment["timestamp"] = pd.to_datetime(particular_experiment["timestamp"])
    
    full_index = pd.MultiIndex.from_product([full_time_range, all_sensors], names=["timestamp", "sensors"])
    extended_particular_experiment= pd.DataFrame(index=full_index).reset_index()
    particular_experiment = extended_particular_experiment.merge(particular_experiment, on=["timestamp", "sensors"], how="left",suffixes=["",None])
    particular_experiment["after_insertion"] = particular_experiment["timestamp"] > turning_point
    particular_experiment["original_value"] = pd.notna(particular_experiment["VOC"])
    dict_of_timeseries[index] = particular_experiment


In [ ]:
example_df = dict_of_timeseries[10]
example_df

In [ ]:
for index, row in userInputData.iterrows():
    
        particular_experiment =  dict_of_timeseries[index]

        particular_experiment["VOC"] = (
            particular_experiment
            .groupby("sensors")["VOC"]
            .transform(lambda s: s.interpolate(method = "cubic"))
        )
        dict_of_timeseries[index] = particular_experiment

In [ ]:
example_df = dict_of_timeseries[10]
example_df

### PROCESS THE TIMESERIES COLUMNS

#### FIND THE MIN AND MAX TIMESTAPS THAT ALL SENSORS SHARE

In [ ]:
# Ensure the columns exist and have datetime dtype
userInputData["actual timestamp StartingExperiment"] = pd.NaT
userInputData["actual timestamp EndingExperiment"] = pd.NaT
userInputData = userInputData.astype({
    "actual timestamp StartingExperiment": "datetime64[ns]",
    "actual timestamp EndingExperiment": "datetime64[ns]"
})

for index, row in userInputData.iterrows():
    particular_experiment = dict_of_timeseries[index]

    first_timestamps = particular_experiment[particular_experiment["VOC"].notna()] \
        .groupby("sensors", as_index=False)["timestamp"].min()
    last_timestamps = particular_experiment[particular_experiment["VOC"].notna()] \
        .groupby("sensors", as_index=False)["timestamp"].max()

    start = first_timestamps["timestamp"].max()
    end = last_timestamps["timestamp"].min()

    userInputData.at[index, "actual timestamp StartingExperiment"] = start
    userInputData.at[index, "actual timestamp EndingExperiment"] = end

    particular_experiment = particular_experiment.loc[
        particular_experiment["timestamp"].between(start, end, inclusive="both")
    ].reset_index(drop=True)

    dict_of_timeseries[index] = particular_experiment


In [ ]:
userInputData.head(20)

In [ ]:
example_df = dict_of_timeseries[10]
example_df

In [ ]:
userInputData

In [ ]:
for index, row in userInputData.iterrows():

    particular_experiment =  dict_of_timeseries[index]
    particular_experiment = particular_experiment.drop_duplicates()
    dict_of_timeseries[index] = particular_experiment

In [ ]:
dict_of_timeseries[7]

#### CREATE TIMEDELTA TYPE COLUMNS FOR TIMESTAMP AND FLOAT COLUMN FOR SECONDS

In [ ]:
userInputData["time taken total"] = pd.NaT

for index, row in userInputData.iterrows():

    particular_experiment =  dict_of_timeseries[index]
    sensors = particular_experiment["sensors"].unique()
    particular_experiment["datetime_timestamp"] = particular_experiment["timestamp"]
    particular_experiment = particular_experiment.drop(columns="timestamp")
    particular_experiment["timestamp"] = np.nan
    particular_experiment["timestamp"] = particular_experiment["timestamp"].astype("timedelta64[ns]")
    for sensor in sensors:
        
        mask = particular_experiment["sensors"]==sensor
        #masked_data = particular_experiment.loc[mask]
        
        start = particular_experiment.loc[mask]["datetime_timestamp"].min()
        end = particular_experiment.loc[mask]["datetime_timestamp"].max()
        
        duration = end - start
        
        timeline = pd.timedelta_range(start="0s", end=duration, freq="s")
        
        particular_experiment.loc[mask,"timestamp"] = timeline
        particular_experiment.loc[mask,"seconds"] = timeline.total_seconds()
        userInputData.at[index,"time taken total"] = duration    
    dict_of_timeseries[index] =   particular_experiment
       


In [ ]:
userInputData

In [ ]:
userInputData.columns

In [ ]:
 dict_of_timeseries[10]

In [ ]:
userInputData.dtypes

In [ ]:
userInputData["timestamp InsertingSource timedelta"] = np.nan
userInputData["timestamp InsertingSource seconds"] = np.nan

for index, row in userInputData.iterrows():

    particular_experiment =  dict_of_timeseries[index]
    sensors = particular_experiment["sensors"].unique()

    for sensor in sensors:
        
        mask = particular_experiment["sensors"]==sensor
        #print(masker)

        start = particular_experiment.loc[mask,:]["timestamp"].min()
       # print(start)
        end = particular_experiment.loc[mask,:]["timestamp"].max()
        #print(end)
       # print(sensor)
        
        mask = (particular_experiment["sensors"] == sensor) & (particular_experiment["after_insertion"] == True)
        filtered = particular_experiment.loc[mask]
        #print(first_index)
        if not filtered.empty:
            first_index = filtered.index[0]
            insertion_time = filtered.at[first_index, "timestamp"]
        
            userInputData.at[index, "timestamp InsertingSource timedelta"] = insertion_time
            userInputData.at[index, "timestamp InsertingSource seconds"] = insertion_time.total_seconds()
            userInputData.at[index, "time taken after insertion"] = (userInputData.at[index, "time taken total"] - insertion_time)
            userInputData.at[index, "time taken before insertion"] = (userInputData.at[index, "time taken total"] - userInputData.at[index, "time taken after insertion"])
            
userInputData["time taken before insertion (seconds)"]  = userInputData["time taken before insertion"].dt.total_seconds() 
userInputData["time taken after insertion (seconds)"]  = userInputData["time taken after insertion"].dt.total_seconds() 
userInputData["time taken total"] = pd.to_timedelta(userInputData["time taken total"], errors="coerce")
userInputData["time taken total (seconds)"] = userInputData["time taken total"].dt.total_seconds() 

In [ ]:
userInputData.head(20)

#### DROP THE EXPERIMENTS WHICH THEY DON'T HAVE ANY ACTUAL VALUE

In [ ]:
# Find indices where the column is NaN
to_drop_idx = userInputData[userInputData["timestamp InsertingSource timedelta"].isna()].index
print(to_drop_idx)
# Remove those entries from the dictionary
for key in to_drop_idx:
    dict_of_timeseries.pop(key, None)

# Drop them from the DataFrame
userInputData = userInputData.dropna(subset=["timestamp InsertingSource timedelta"])


In [ ]:


#if set is empty, the deletion at dictionary was succesful
missing_from_dict = set(userInputData.index) - set(dict_of_timeseries.keys())
print("Keys missing from dictionary:", missing_from_dict)



## SEARCH THE DATA AND CONTINUE THE PREPOSSESSING

In [ ]:
userInputData["room"].unique()

In [ ]:

#open_door_mask = userInputData["are-doors-opened"] != "on"
exp_state_mask = (userInputData["experimentState"] =="InsertingSourcePollutant")
#mask = room_mask & open_door_mask & exp_state_mask
cond_mask = exp_state_mask
room_list = userInputData["room"].unique()
for room in room_list:
    
userInputDataCheck = userInputData.copy()
    
    value = 240  # your threshold value
    percentage = (userInputDataCheck["time taken after insertion (seconds)"] >= value).mean() * 100
    print(f"{percentage:.2f}% of rows have time taken after insertion (seconds) > {value}")
    
    value = 300  # your threshold value
    percentage = (userInputDataCheck["time taken after insertion (seconds)"] >= value).mean() * 100
    print(f"{percentage:.2f}% of rows have time taken after insertion (seconds) > {value}")
    
    value = 360  # your threshold value
    percentage = (userInputDataCheck["time taken after insertion (seconds)"] >= value).mean() * 100
    print(f"{percentage:.2f}% of rows have time taken after insertion (seconds) < {value}")

In [ ]:
#room_mask = userInputData["room"].isin(['Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0', 'Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 , Δ:2.00'])

#open_door_mask = userInputData["are-doors-opened"] != "on"
exp_state_mask = (userInputData["experimentState"] =="InsertingSourcePollutant")
#mask = room_mask & open_door_mask & exp_state_mask
mask = exp_state_mask
userInputDataCheck = userInputData.copy()

value = 30  # your threshold value
percentage = (userInputDataCheck["time taken before insertion (seconds)"] >= value).mean() * 100
print(f"{percentage:.2f}% of rows have time taken before insertion (seconds) >= {value}")

value = 60  # your threshold value
percentage = (userInputDataCheck["time taken before insertion (seconds)"] >= value).mean() * 100
print(f"{percentage:.2f}% of rows have time taken before insertion (seconds) >= {value}")

value = 120  # your threshold value
percentage = (userInputDataCheck["time taken before insertion (seconds)"] >= value).mean() * 100
print(f"{percentage:.2f}% of rows have time taken before insertion (seconds) >= {value}")

In [ ]:
value=60
userInputDataCheck.loc[mask & (userInputDataCheck["time taken before insertion (seconds)"] < value)]

In [ ]:
#if set is empty, the deletion at dictionary was succesful
missing_from_dict = set(userInputData.index) - set(dict_of_timeseries.keys())
print("Keys missing from dictionary:", missing_from_dict)


In [ ]:
userInputData

In [ ]:
for index, row in userInputData.iterrows():
    
    particular_experiment =  dict_of_timeseries[index]
    particular_experiment["seconds passed from insertionSource"] = np.nan
    sensors = particular_experiment["sensors"].unique()
    
    for sensor in sensors:
        
        mask = particular_experiment["sensors"]==sensor
        particular_experiment.loc[mask,"seconds passed from insertionSource"] = particular_experiment.loc[mask,"seconds"] - row["timestamp InsertingSource seconds"]  
    dict_of_timeseries[index] =   particular_experiment


In [ ]:
dict_of_timeseries[10]

In [ ]:
print(dict_of_timeseries[157].max())
print(userInputData.loc[157])

In [ ]:
#make a column from VOC with the min at zerofor index, row in userInputData.iterrows():
for index, row in userInputData.iterrows():

    particular_experiment =  dict_of_timeseries[index].copy()
    col_to_create = "VOC original"
    col_name = "VOC"
    particular_experiment[col_to_create] = particular_experiment[col_name].copy()
    sensors = particular_experiment["sensors"].unique()
    if col_name  not in particular_experiment.columns:
        particular_experiment[col_name] = np.nan
    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor
        particular_experiment.loc[mask,col_name] = particular_experiment.loc[mask,col_name] - particular_experiment.loc[mask,col_name].min()
    dict_of_timeseries[index]  =  particular_experiment
   

In [ ]:

#use rolling window
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index].copy()
    sensors = particular_experiment["sensors"].unique()

    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor
        particular_experiment.loc[mask,"VOC rolling average"] = particular_experiment.loc[mask].rolling(30,center= True ,min_periods=1,on="seconds")["VOC"].mean()


        
 #   scaler = StandardScaler()
#    particular_experiment.loc[:,"VOC standard scaled"] = scaler.fit_transform(particular_experiment.loc[:,"VOC"].to_numpy().reshape(-1,1))
#    particular_experiment.loc[:,"VOC rolling average standard scaled"] = scaler.fit_transform(particular_experiment.loc[:,"VOC rolling average"].to_numpy().reshape(-1,1))


    dict_of_timeseries[index] = particular_experiment  

In [ ]:
#round VOC rolling average data at 3 decimals
for index, row in userInputData.iterrows():
    
    particular_experiment =  dict_of_timeseries[index].copy()
    particular_experiment["VOC"] = particular_experiment["VOC"].round(3)
    particular_experiment["VOC rolling average"] = particular_experiment["VOC rolling average"].round(4)
    dict_of_timeseries[index]  =  particular_experiment

In [ ]:
#trim the time before insertion to be less than two minutes 
#trim the time after insertion to be less than ten minutes
max_before= 31
max_after = 300
columns_to_add = ["time taken total (seconds)-capped","time taken before insertion (seconds)-capped","time taken after insertion (seconds)-capped"]
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index]
    for col in columns_to_add:
        if col not in userInputData.columns:
            userInputData.loc[:,col] = np.nan  
      
    time_mask = (particular_experiment["seconds passed from insertionSource"] > ( - max_before)) & (particular_experiment["seconds passed from insertionSource"] < max_after)
    particular_experiment = particular_experiment.loc[time_mask,:].copy()
   
    
    particular_experiment["original seconds"] = particular_experiment["seconds"]
    sensors = particular_experiment["sensors"].unique()
    
    for sensor in sensors:
        mask = particular_experiment["sensors"] == sensor 
        min_sensor_time = particular_experiment.loc[mask,"seconds passed from insertionSource"].min()
        max_sensor_time = particular_experiment.loc[mask,"seconds passed from insertionSource"].max()
        particular_experiment.loc[mask,"seconds"] = np.arange(min_sensor_time,max_sensor_time + 1) 
    min_sensor_time = particular_experiment["seconds passed from insertionSource"].min()    
    max_sensor_time = particular_experiment["seconds passed from insertionSource"].max()    

    userInputData.at[index,"time taken before insertion (seconds)-capped"] = min_sensor_time
    userInputData.at[index,"time taken after insertion (seconds)-capped"] =max_sensor_time
    userInputData.at[index,"time taken total (seconds)-capped"] = min_sensor_time +  max_sensor_time +1

    dict_of_timeseries[index]  =  particular_experiment


In [ ]:
dict_of_timeseries[11]

In [ ]:
print(dict_of_timeseries[157].max())
print(userInputData.loc[157])

In [ ]:
dict_of_timeseries[12]

In [ ]:
userInputData.loc[:,["front-wall","side-right-wall","back-wall","side-left-wall"]] = userInputData.loc[:,["front-wall","side-right-wall","back-wall","side-left-wall"]].round(1)

In [ ]:
userInputData

#make for each experiment and sensor as min the value zero
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index].copy()
    sensors = particular_experiment["sensors"].unique()
    col_name = "VOC rolling average"
    if col_name  not in particular_experiment.columns:
        particular_experiment[col_name] = np.nan
    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor
        particular_experiment.loc[mask,col_name] = particular_experiment.loc[mask,"VOC rolling average"] - particular_experiment.loc[mask,"VOC rolling average"].min()
    dict_of_timeseries[index]  =  particular_experiment
   

In [ ]:
dict_of_timeseries[157]

In [ ]:
dict_of_timeseries[157]

In [ ]:
#make the seconds to be index
for index, row in userInputData.iterrows():
    
    particular_experiment =  dict_of_timeseries[index].copy()
    
    dict_of_timeseries[index]  =  particular_experiment.set_index("seconds",drop = False)



In [ ]:
dict_of_timeseries[190]

In [ ]:
#find max values and its position
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index].copy()
    sensors = particular_experiment["sensors"].unique()
    column_to_find_values = "VOC rolling average"
    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor
        
        col_name = f"{sensor} MAX value {column_to_find_values}"
        if col_name  not in userInputData.columns:
            userInputData.loc[:,col_name] = np.nan
        userInputData.at[index,col_name] = particular_experiment.loc[mask,column_to_find_values].max()
    
            
        col_name = f"{sensor} index of MAX value {column_to_find_values}"
        if col_name  not in userInputData.columns:
            userInputData.loc[:,col_name] = np.nan    
        userInputData.at[index,col_name] = particular_experiment.loc[mask,column_to_find_values].idxmax()

dict_of_timeseries[index]  =  particular_experiment


In [ ]:
userInputData

In [ ]:
print(dict_of_timeseries[157].max())
print(userInputData.loc[157])

In [ ]:
col_list = []

for sensor in sensors:
    col_name = f"{sensor} MAX value {column_to_find_values}"
    col_list.append(col_name)
colunn_to_create = "sensor with MAX value experiment"    
if colunn_to_create not in userInputData.columns:
    userInputData[colunn_to_create] = np.nan
colunn_to_create_second = "sensor with second MAX value experiment"
if colunn_to_create not in userInputData.columns:
    userInputData[colunn_to_create] = np.nan
for index, row in userInputData.iterrows():
    max_dict = row[col_list].to_dict()  
    # Sort correctly — key must be inside sorted()
    sorted_items = sorted(
    max_dict.items(),
    key=lambda x: (pd.notna(x[1]), x[1])
)
    # Handle missing/NaN gracefully
    if len(sorted_items) > 0 and pd.notna(sorted_items[2][1]):
        userInputData.at[index, colunn_to_create] = sorted_items[2][0]
    if len(sorted_items) > 1 and pd.notna(sorted_items[1][1]):
        userInputData.at[index, colunn_to_create_second] = sorted_items[1][0]


In [ ]:
#FIND 90% quantile and 99% quantile not yet use at capping values
#find quantile per sensor 
def create_column_quantile(userInputData,sensor,quantile_number,column_to_find_values):
    colunn_to_create = f"{sensor} quantile {quantile_number * 100}% {column_to_find_values}"    
    if colunn_to_create not in userInputData.columns:
        userInputData[colunn_to_create] = np.nan
    return colunn_to_create

    
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index].copy()

    column_to_find_values = "VOC rolling average"
    sensors = particular_experiment["sensors"].unique()

    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor

        quantile_number = 0.9
        colunn_to_create = create_column_quantile(userInputData,sensor,quantile_number,column_to_find_values)
    
        userInputData.at[index, colunn_to_create] = particular_experiment.loc[mask,column_to_find_values].quantile(quantile_number)

        quantile_number = 0.99
        colunn_to_create = create_column_quantile(userInputData,sensor,quantile_number,column_to_find_values)
    
        userInputData.at[index, colunn_to_create] = particular_experiment.loc[mask,column_to_find_values].quantile(quantile_number)


        
    dict_of_timeseries[index]  =   particular_experiment

In [ ]:
#FIND 90% quantile and 99% quantile not yet use at capping values
#find quantile per experiment
def create_column_quantile(userInputData,sensor,quantile_number,column_to_find_values):
    colunn_to_create = f"{sensor} quantile {quantile_number * 100}% {column_to_find_values}"    
    if colunn_to_create not in userInputData.columns:
        userInputData[colunn_to_create] = np.nan
    return colunn_to_create

    
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index].copy()

    column_to_find_values = "VOC rolling average"
    sensor = "whole experiment"


    quantile_number = 0.9
    colunn_to_create = create_column_quantile(userInputData,sensor,quantile_number,column_to_find_values)

    userInputData.at[index, colunn_to_create] = particular_experiment.loc[:,column_to_find_values].quantile(quantile_number)

    quantile_number = 0.99
    colunn_to_create = create_column_quantile(userInputData,sensor,quantile_number,column_to_find_values)

    userInputData.at[index, colunn_to_create] = particular_experiment.loc[:,column_to_find_values].quantile(quantile_number)


        
    dict_of_timeseries[index]  =   particular_experiment

In [ ]:
userInputData

In [ ]:
userInputData.head(20)

In [ ]:
#cap by max value 
offset_to_increase_capped = 5
for index, row in userInputData.iterrows():
    
    particular_experiment =  dict_of_timeseries[index].copy()
    colunn_to_create = "VOC-capped"
    if colunn_to_create not in userInputData.columns:
        particular_experiment[colunn_to_create] = np.nan
        
    max_sensor = row["sensor with MAX value experiment"]

    max_sensor = max_sensor.split()[0]

    max_sensor_column_name = f"{max_sensor} MAX value VOC rolling average"
    max_value = row[max_sensor_column_name]
    
    second_max_sensor = row["sensor with second MAX value experiment"].split()[0]
    
    second_max_sensor_column_name = f"{second_max_sensor} MAX value VOC rolling average"
    second_max_value = row[second_max_sensor_column_name]  
    value_to_cap_into =  5  + second_max_value
   
        
    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor
        particular_experiment.loc[mask,colunn_to_create] = particular_experiment.loc[mask,"VOC"]

        if (sensor == max_sensor):
            full_mask = mask & (particular_experiment["VOC"]> value_to_cap_into)
            particular_experiment.loc[full_mask,colunn_to_create] = value_to_cap_into
        
    dict_of_timeseries[index]  =   particular_experiment

In [ ]:
#cap by max value 
offset_to_increase_capped = 5
for index, row in userInputData.iterrows():
    
    particular_experiment =  dict_of_timeseries[index].copy()
    colunn_to_create = "VOC rolling average-capped"
    if colunn_to_create not in userInputData.columns:
        particular_experiment[colunn_to_create] = np.nan
        
    max_sensor = row["sensor with MAX value experiment"]

    max_sensor = max_sensor.split()[0]

    max_sensor_column_name = f"{max_sensor} MAX value VOC rolling average"
    max_value = row[max_sensor_column_name]
    
    second_max_sensor = row["sensor with second MAX value experiment"].split()[0]
    
    second_max_sensor_column_name = f"{second_max_sensor} MAX value VOC rolling average"
    second_max_value = row[second_max_sensor_column_name]  
    value_to_cap_into =  5  + second_max_value
   
        
    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor
        particular_experiment.loc[mask,colunn_to_create] = particular_experiment.loc[mask,"VOC rolling average"]

        if (sensor == max_sensor):
            full_mask = mask & (particular_experiment["VOC rolling average"]> value_to_cap_into)
            particular_experiment.loc[full_mask,colunn_to_create] = value_to_cap_into
        
    dict_of_timeseries[index]  =   particular_experiment

In [ ]:
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index].copy()
    sensors = particular_experiment["sensors"].unique()

    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor
        column_to_create = "VOC gradient"
        if column_to_create not in particular_experiment.columns:
            particular_experiment.loc[:,column_to_create] = np.nan
        particular_experiment.loc[mask,column_to_create] = np.gradient(particular_experiment.loc[mask,"VOC rolling average"].to_numpy())
        
        
        mask = particular_experiment["sensors"]==sensor
        column_to_create = "VOC rolling average gradient"
        if column_to_create not in particular_experiment.columns:
            particular_experiment.loc[:,column_to_create] = np.nan
        particular_experiment.loc[mask,column_to_create] = np.gradient(particular_experiment.loc[mask,"VOC rolling average"].to_numpy())
        
        # column_to_create = "VOC rolling average-capped gradient"
        # if column_to_create not in particular_experiment.columns:
        #     particular_experiment.loc[:,column_to_create] = np.nan
        #particular_experiment.loc[mask,column_to_create] = np.gradient(particular_experiment.loc[mask,"VOC rolling average-capped"].to_numpy())

    dict_of_timeseries[index] = particular_experiment      

In [ ]:
userInputData

In [ ]:
#find the VOC values and gradient mean in calm periods,both in NoSourcePollutantInserted and before insertion source
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index].copy()
    sensors = particular_experiment["sensors"].unique()

    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor
        #if the experiment had a pollutant source, grab only the rows with negative seconds
        if userInputData.at[index,"experimentState"] == "InsertingSourcePollutant":
            mask = mask & (particular_experiment["seconds"]<0)
            
        column_to_use = "VOC"  
        column_to_insert = f"{sensor} {column_to_use} mean calm period"
        if column_to_insert not in userInputData.columns:
            userInputData[column_to_insert] = np.nan
        userInputData.at[index,column_to_insert] = particular_experiment.loc[mask,column_to_use].mean()
        
        column_to_use = "VOC gradient"  
        column_to_insert = f"{sensor} {column_to_use} mean calm period"
        if column_to_insert not in userInputData.columns:
            userInputData[column_to_insert] = np.nan
        userInputData.at[index,column_to_insert] = particular_experiment.loc[mask,column_to_use].mean()    


In [ ]:
#create standard scaler per experiment and per sensor, per VOC rolling average and VOC rolling average capped

#per sensor 
from sklearn.preprocessing import StandardScaler
def create_column_standard_scaler(particular_experiment,column_to_find_values):
    
    colunn_to_create = f"{column_to_find_values} standard scaler per sensor"    
    if colunn_to_create not in particular_experiment.columns:
        particular_experiment[colunn_to_create] = np.nan
    return colunn_to_create
    
columns_to_find_values = ["VOC rolling average","VOC rolling average-capped"]

for column_to_find_values in columns_to_find_values:
    for index, row in userInputData.iterrows():
    
        particular_experiment =  dict_of_timeseries[index].copy()
        scaler = StandardScaler()
        sensors = particular_experiment["sensors"].unique()
        for sensor in sensors:
            
            mask = particular_experiment["sensors"]==sensor
            colunn_to_create = create_column_standard_scaler(particular_experiment,column_to_find_values)
        
            particular_experiment.loc[mask, colunn_to_create] = scaler.fit_transform( (particular_experiment.loc[mask,column_to_find_values].to_numpy().reshape(-1,1)))
    
        dict_of_timeseries[index]  =   particular_experiment

In [ ]:
#experiment
from sklearn.preprocessing import StandardScaler
def create_column_standard_scaler(particular_experiment,column_to_find_values):
    
    colunn_to_create = f"{column_to_find_values} standard scaler whole experiment"    
    if colunn_to_create not in particular_experiment.columns:
        particular_experiment[colunn_to_create] = np.nan
    return colunn_to_create
    
columns_to_find_values = ["VOC rolling average","VOC rolling average-capped"]

for column_to_find_values in columns_to_find_values:
    for index, row in userInputData.iterrows():
    
        particular_experiment =  dict_of_timeseries[index].copy()
        scaler = StandardScaler()
            
        colunn_to_create = create_column_standard_scaler(particular_experiment,column_to_find_values)
    
        particular_experiment.loc[:, colunn_to_create] = scaler.fit_transform((particular_experiment.loc[:,column_to_find_values].to_numpy().reshape(-1,1)))
    
        dict_of_timeseries[index]  =   particular_experiment

In [ ]:
dict_of_timeseries[157]

In [ ]:
room_length = 4.0
room_width = 3.25
userInputData["x axis"] = np.nan
userInputData["y axis"] = np.nan
#front is max,back is zero,right is max,left is zero
def find_x_axis(x,room_width):
    from_right = None
    from_left = None

    #return NaN either if numbers at the position columns are negative or above room width
    if (x["side-right-wall"]<0 and x["side-right-wall"]> room_width) or (x["side-left-wall"]< 0 and x["side-left-wall"]> room_width) or (pd.isna(x["side-right-wall"]) and pd.isna(x["side-left-wall"])):
        x["x axis"] = np.nan
        return x["x axis"] 
    if pd.notna(x["side-right-wall"]):
       from_right = -x["side-right-wall"]
    if pd.notna(x["side-left-wall"]):
       from_left = -room_width + x["side-left-wall"]
      
    if  from_right is not None and from_left is not None:
        if from_right == from_left:
            x["x axis"] = from_right
        else:
            x["x axis"] = np.nan
          

    elif pd.notna(x["side-left-wall"]):
    
        x["x axis"] = from_left

    elif pd.notna(x["side-right-wall"]):
        x["x axis"] = from_right
    return  x["x axis"]

def find_y_axis(x,room_length):
    from_front = None
    from_back = None

    #return NaN either if numbers at the position columns are negative or above room width
    if (x["front-wall"]<0 and x["front-wall"]> room_length) or (x["back-wall"]< 0 and x["back-wall"]> room_length) or (pd.isna(x["front-wall"]) and pd.isna(x["back-wall"])):
        x["y axis"] = np.nan
        return x["y axis"] 
    if pd.notna(x["front-wall"]):
       from_front = room_length - x["front-wall"]
    if pd.notna(x["back-wall"]):
       from_back = x["back-wall"]
    if from_front is not None and from_back is not None:
        if from_front == from_back:
            x["y axis"] = from_front
        else:
            x["y axis"] = np.nan
            
    elif pd.notna(x["back-wall"]):
    
        x["y axis"] = from_back

    elif pd.notna(x["front-wall"]):
        x["y axis"] = from_front

    return  x["y axis"]

userInputData["x axis"] = userInputData.apply(lambda x:find_x_axis(x,room_width),axis=1)
userInputData["y axis"] = userInputData.apply(lambda x:find_y_axis(x,room_length),axis=1)

In [ ]:
userInputData.head(30)

In [ ]:
userInputData.loc[:, ["x axis", "y axis"]].tail(30)

In [ ]:
print(dict_of_timeseries[157].max())
print(userInputData.loc[157])

In [ ]:
userInputData["room"].unique()

In [ ]:
room_length = 4.0
room_width = 3.25
pos_id0 = 'position of Id=0:BME680:breathVocEquivalent'
pos_id1 = 'position of Id=1:BME680:breathVocEquivalent'
pos_id2 = 'position of Id=2:BME680:breathVocEquivalent'
pos_id_all = [pos_id0,pos_id1,pos_id2]
room_positions = {
    'Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55' : {
        pos_id1 +"-x axis" : room_width - 0.85 ,
        pos_id1 +"-y axis" : room_length - 0.95,
        pos_id2+"-x axis" :  room_width - 1.55 ,
        pos_id2+  "-y axis" : room_length - 0.95
              
    },
        
    'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1' : {
        pos_id1+"-x axis" :  room_width - 1.2 ,
        pos_id1+"-y axis" :  0.6,
        pos_id2+"-x axis" :  room_width -1.1 ,
        pos_id2+"-y axis" : room_length - 0.8
   
    },      
        
    'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90' : {
        
        pos_id0+"-x axis" : -room_width + 0.9, 
        pos_id0+"-y axis" : room_length - 0.8,
        
        pos_id1+"-x axis" : -room_width + 0.9,
        pos_id1+"-y axis" : room_length - 0.8,
        
        pos_id2+"-x axis" : -room_width + 0.9,
        pos_id2+"-y axis" : room_length - 0.8
     },
    'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0' :{
        pos_id0+"-x axis" : -room_width  + 1.4, 
        pos_id0+"-y axis" : room_length - 0.5,
        
        pos_id1+"-x axis" :   -0.5,
        pos_id1+"-y axis" :   1.7,
        
        pos_id2+"-x axis" :  -room_width +1.0,
        pos_id2+"-y axis" :  0.4    
    }
}
room_positions

In [ ]:
pos_id0 = 'position of Id=0:BME680:breathVocEquivalent'
pos_id1 = 'position of Id=1:BME680:breathVocEquivalent'
pos_id2 = 'position of Id=2:BME680:breathVocEquivalent'
pos_id_all = [pos_id0,pos_id1,pos_id2]
for pos_id in pos_id_all:
    userInputData[pos_id+"-x axis"] = np.nan
    userInputData[pos_id+"-y axis"] = np.nan
    


for room,sensor_positions in room_positions.items():
   # print(f"room:{room}")
    outer_mask = userInputData["room"] == room
  #  print(f"sensor_positions:{sensor_positions}")
    for sensor_position,value  in sensor_positions.items():
 
        userInputData.loc[outer_mask,sensor_position] = value

In [ ]:
userInputData

In [ ]:
#find euclidian distance between sensors position and source

x_axis_source = userInputData["x axis"].to_numpy()
y_axis_source = userInputData["y axis"].to_numpy()


for id_sensor_number in range(3):
    sensor_name = "Id="+str(id_sensor_number)+":BME680:breathVocEquivalent"
    x_axis_sensor = userInputData["position of "+ sensor_name + "-x axis"].to_numpy()
    y_axis_sensor = userInputData["position of "+ sensor_name + "-y axis"].to_numpy()
    # Euclidean distance per row
    distances = np.sqrt((x_axis_sensor - x_axis_source)**2 + (y_axis_sensor - y_axis_source)**2)
    var_name = "Euclidian distance to "+sensor_name
    userInputData[var_name] = distances
    #round the distance
    userInputData[var_name] = userInputData[var_name].round(2)
    

In [ ]:
userInputData

In [ ]:
userInputData.loc[:, ["x axis", "y axis"]]

In [ ]:
print(dict_of_timeseries[157].max())
print(userInputData.loc[157])

from matplotlib.ticker import FuncFormatter

def createTheTitleForTheFigure(experiment_number):
    date = userInputData.iloc[experiment_number]["timestamp InsertingSource"].date()
    row  = userInputData.iloc[experiment_number]
    if ((experiment_number > 0 )):
        previous_date = userInputData.iloc[experiment_number - 1]["timestamp InsertingSource"].date()
        if (date != previous_date):
            print("........................................................\n\n")
    
    title = "Experiment at row " + str(experiment_number)+ "At "+date.strftime("%Y-%m-%d")+ f" BME680:breathVocEquivalent.\n"
    if (userInputData.iloc[experiment_number]["experimentState"] == "InsertingSourcePollutant"):
        title= title + "at the pollution source position to be "
        if (pd.isna(userInputData.iloc[experiment_number]['front-wall'])==False):
           title =title + f"{userInputData.iloc[experiment_number]['front-wall']} meters from the front wall, "
        if (pd.isna(userInputData.iloc[experiment_number]['back-wall'])==False):
           title =title +f"{userInputData.iloc[experiment_number]['back-wall']} meters from the back wall, "
        if (pd.isna(userInputData.iloc[experiment_number]['side-right-wall'])==False):
           title =title+ f"{userInputData.iloc[experiment_number]['side-right-wall']} meters from the side right wall, "
        if (pd.isna(userInputData.iloc[experiment_number]['side-left-wall'])==False):
           title =title+ f"{userInputData.iloc[experiment_number]['side-left-wall']} meters from the side left wall, "
    elif (userInputData.iloc[experiment_number]["experimentState"] == "NoSourcePollutantInserted"):  
        title =title + "without source insertion.\n"
        
        
    
    title =title +"at the room:"+userInputData.iloc[experiment_number]["room"]+"\n"   
    title=title +"Στοιχεία για το μενωμένο πείραμα:"
    title= title+"Αντικείμενο που χρησιμοποιείται¨"+row["item-used"]+", "
    if (pd.isna(row["are-doors-opened"]) == False):
        title = title + "Οι πόρτες είναι ανοιχτές, "
    if (pd.isna(row["are-fans-on"]) == False):
        title = title + "Οι ανεμιστήρες είναι ενεργοποιημένοι, "
    if (pd.isna(row["are-people-inside"]) == False):
        title = title + "Βρίσκονται άνθρωποι μέσα, "
    if (pd.isna(row["are-windows-opened"]) == False):
        title = title + "Τα παράθυρα είναι ανοιχτά, "
    if (pd.isna(row["notes"])== False):
        title= title + "Σημειώσεις:"+ row["notes"]

    return title

def printData(dict_of_timeseries,plot_old_data = True):
    for experiment_number in dict_of_timeseries:
        particular_experiment =  dict_of_timeseries[experiment_number]
        
        title = createTheTitleForTheFigure(experiment_number)
            
       
        plt.figure(figsize=(18, 8))
        if plot_old_data:
            y_column_data = "VOC"
        else:
            y_column_data = "cutted_VOC"
            
        sns.lineplot(data=particular_experiment, x="seconds", y=y_column_data, hue="sensors")
        def format_seconds(x, _):
            return str(pd.to_timedelta(x, unit="s")).split()[-1]  # hh:mm:ss only
        plt.gca().xaxis.set_major_formatter(FuncFormatter(format_seconds))
        plt.xticks(rotation=45)
        
        plt.axvline(x=userInputData.at[experiment_number,"timestamp InsertingSource seconds"], color="red", linestyle="--", linewidth=2, label="timestamp of the inserting source")
        plt.title(title)
    
        plt.show()
    
        print("\n")

In [ ]:
#printData(dict_of_timeseries,plot_old_data = True)

In [ ]:
userInputData

In [ ]:
dict_of_timeseries[12]

In [ ]:
#create a huge dataframe to save it 

# Combine into one DataFrame, keeping the dictionary key
timeSeriesData_BIG = pd.concat(dict_of_timeseries, names=["keys"])

# Reset index so the key becomes a column
timeSeriesData_BIG = timeSeriesData_BIG.reset_index(level="keys").reset_index(drop=True)
timeSeriesData_BIG["seconds"] = timeSeriesData_BIG["seconds"].astype(int)
timeSeriesData_BIG = timeSeriesData_BIG.set_index("seconds",drop=False)

In [ ]:
timeSeriesData_BIG

In [ ]:
print(timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"] == 157].max())
print(userInputData.loc[157])


In [ ]:
userInputData.loc[:,"timestamp InsertingSource timedelta"] = userInputData["timestamp InsertingSource timedelta"].astype("timedelta64[ns]")

In [ ]:
userInputData.dtypes

In [ ]:
userInputData

def find_column_values_passed_by_function(timeSeriesData_BIG,userInputData,column,callback_function):
    sensors = timeSeriesData_BIG["sensors"].unique()
    #create columns first if they don't exist for the UserInputDAta
    for sensor in sensors:
        
        col_name_before = f"{sensor} before insertion {str(callback_function.__name__)} {column}"
        
        if col_name_before not in userInputData.columns:
            userInputData.loc[:,col_name_before] = np.nan
            
        col_name_after = f"{sensor} after insertion {str(callback_function.__name__)} {column}" 
    
        if col_name_after not in userInputData.columns:
            userInputData.loc[:,col_name_before] = np.nan
            
        callback_function_before_insertion ={}
        callback_function_after_insertion = {}
            # Define masks
        mask_before = timeSeriesData_BIG["seconds passed from insertionSource"] < 0
        mask_after = ~mask_before
    
       # ---- Before insertion ----
        series_before = (
            timeSeriesData_BIG[mask_before & (timeSeriesData_BIG["sensors"] == sensor)]
            .groupby("keys")[column].apply(lambda x:callback_function(x))
        )
        # ---- After insertion ----
        series_after = (
            timeSeriesData_BIG[mask_after & (timeSeriesData_BIG["sensors"] == sensor)]
            .groupby("keys")[column].apply(lambda x:callback_function(x))
        )
        userInputData.loc[:,col_name_before] = series_before
        userInputData.loc[:,col_name_after] = series_after


    return userInputData  

def find_diff_between_two_columns_created_values(timeSeriesData_BIG,userInputData,column,callback_function_one,callback_function_second):
    sensors = timeSeriesData_BIG["sensors"].unique()
    for sensor in sensors:
        
        col_name_before = [f"{sensor} before insertion {str(callback_function_one.__name__)} {column}", f"{sensor} before insertion {str(callback_function_second.__name__)} {column}"]
        col_name_after = [f"{sensor} after insertion {str(callback_function_one.__name__)} {column}",f"{sensor} after insertion {str(callback_function_second.__name__)} {column}" ]
        col_name_before_diff = f"{sensor} before insertion diff between {str(callback_function_one.__name__)} and {str(callback_function_second.__name__)} {column}"
        col_name_after_diff = f"{sensor} before insertion diff between {str(callback_function_one.__name__)} and {str(callback_function_second.__name__)} {column}"
        userInputData.loc[:,col_name_before_diff] = userInputData[col_name_before[1]] - userInputData[col_name_before[0]]
        userInputData.loc[:,col_name_after_diff] = userInputData[col_name_after[1]] - userInputData[col_name_after[0]]
    return userInputData  

columns = ["VOC","VOC rolling average"]
callback_function = [np.mean,np.median]
for column in columns:
    userInputData = find_column_values_passed_by_function(timeSeriesData_BIG,userInputData,column,callback_function[0])
    userInputData = find_column_values_passed_by_function(timeSeriesData_BIG,userInputData,column,callback_function[1])
    userInputData = find_diff_between_two_columns_created_values(timeSeriesData_BIG,userInputData,column,callback_function[0],callback_function[1])

In [ ]:
userInputData

In [ ]:
userInputData.columns

In [ ]:
timeSeriesData_BIG

In [ ]:
def saveDataIntoDataFolder(data,data_file_name):
    script_dir = Path().resolve().parent
    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (data_file_name + ".json")
    with open(file_path, 'w') as file:
        if isinstance(data, pd.DataFrame):
            print("It's a DataFrame")
            if  data.empty:
                print("No data to save.")
            else:
                data.to_json(file_path, orient='records', lines=False)             
                print(f"Data saved to {file_path}")

        else:  
            print("It's NOT a DataFrame.")    
            if not data:
                print("No data to save.")
            else:    
                json.dump(data, file)
                print(f"Data saved to {file_path}")


saveDataIntoDataFolder(userInputData,"UserPrevious experiments-preprocessed")
saveDataIntoDataFolder(timeSeriesData_BIG,"Data:Previous experiments-preprocessed")